In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [ ]:
def compute_theta_t(params, t):
    global kappa1, kappa2
    V, V_prime, theta = params
    theta_t = theta * t + (V - theta) * (1-np.exp(-kappa1 * t)) / kappa1 + (V_prime - theta) * kappa1 * ((1-np.exp(-kappa2 * t)) / kappa2 - (1-np.exp(-kappa1 * t)) / kappa1 ) / (kappa1 - kappa2)
    return theta_t

def compute_phi(params, t):
    global params_theta_t
    rho, eta, gamma = params
    theta_t = compute_theta_t(params_theta_t, t)
    return eta / theta_t ** (gamma)

def compute_ssvi_inference(params_w_t, params_theta_t, t, k):
    rho, eta, gamma = params_w_t
    theta_t = compute_theta_t(params_theta_t, t)
    phi = compute_phi(params_w_t, t)
    return 0.5 * theta_t * ( 1 + rho * phi * k + np.sqrt((phi * k + rho)**2 + 1 - rho **2 ))

In [ ]:
tmp = pd.read_csv("clean_data2023.csv", parse_dates=['QUOTE_DATE', 'EXPIRE_DATE'])
tmp

In [ ]:
tmp_iv = tmp[(tmp['QUOTE_DATE'] == "2023-01-04")&(tmp['DTE'] > 14)]

In [ ]:
x, y = np.meshgrid(np.arange(tmp_iv['Log-Strike'].min(), tmp_iv['Log-Strike'].max(), 0.02), np.arange(15/365, tmp_iv['DTE'].max()/365, 0.05))

kappa1 = 5.5
kappa2 = 0.1

################################# LIST OF PARAMETERS FOR EACH DAY #############################################

# Format: V, V_prime, theta, rho, eta, gamma => Date
# 4.196e-02  4.059e-02  1.100e-01, -0.57642764,  0.55170206,  0.6895538 => 2023-01-04
# 0.03897386, 0.03987186, 0.10999981, -0.65123362,  0.41200321,  0.74083532 => 2023-03-07
# 0.02692915, 0.0373211 , 0.10999473, -0.64939349  0.60386782  0.69145943 => 2023-05-04
# 0.01355346, 0.03302254, 0.10998425, -0.66607082,  0.53899324,  0.73193646 => 2023-08-04
# rho, eta, gamma
params_w_t = np.array([-0.57642764,  0.55170206,  0.6895538])# <= you need to manually plug in the parameters
# V, V_prime, theta
params_theta_t = np.array([4.196e-02,  4.059e-02,  1.100e-01])# <= you need to manually plug in the parameters

###############################################################################################################

wt = compute_ssvi_inference(params_w_t, params_theta_t, y, x)# This function outputs the IV from SSVI
sigma_hat = (wt / y) ** 0.5# Convert to IV
sigma_hat.shape

In [ ]:
fig = go.Figure(data=[go.Surface(z=sigma_hat, x=x, y=y)])
fig.update_layout(title='SSVI IV', autosize=False,
                  width=500, height=500,
                  margin=dict(l=65, r=50, b=65, t=90))
fig.update_layout(scene=dict(xaxis_title='Log-Strike', yaxis_title='DTE', zaxis_title='IV'), autosize=True)
fig.show()